## This Notebook generates Table1 report.

In [1]:
%load_ext rpy2.ipython

Error importing in API mode: ImportError('On Windows, cffi mode "ANY" is only "ABI".')
Trying to import in ABI mode.


## Table 1 based on Rubins Ruls

In [ ]:
%%R
library(mice)
library(dplyr)
library(haven)
library(openxlsx)

# ==============================================================================
# 0. Load data and apply inclusion/exclusion criteria on raw dataset
# ==============================================================================
archive_path <- "..."
df_raw <- read_sav(archive_path)

# Age filter (only individuals aged 20 years and older)
df <- df_raw[!is.na(df_raw$Age) & df_raw$Age >= 20, ]

# Define variable groups based on distribution assumptions
normal_vars      <- c("Mean_Calcium", "Mean_Magnesium", "BMI")
non_normal_vars  <- c("Mean_Phosphor", "Age", "Sofa_Mean", "Apache")
binary_vars      <- c("Sex", "Smoky", "HTN", "DM", "IHD", "CVA", 
                      "Pulmonary_Insufficiency", "Brain_Tumor", "Ventilation_Status")
categorical_vars <- c("Diagnosis")
target_var       <- "Status"

df[[target_var]] <- factor(df[[target_var]], levels = c(1, 2), labels = c("Death", "Discharge"))

# Extract actual sample size dynamically for column headers
n_death     <- sum(df[[target_var]] == "Death", na.rm = TRUE)
n_discharge <- sum(df[[target_var]] == "Discharge", na.rm = TRUE)

df[normal_vars]     <- lapply(df[normal_vars], as.numeric)
df[non_normal_vars] <- lapply(df[non_normal_vars], as.numeric)
df[binary_vars]     <- lapply(df[binary_vars], as.factor)
df[categorical_vars]<- lapply(df[categorical_vars], as.factor)

# Run MICE on filtered dataset
cat("-> Running MICE for Comprehensive Table 1...\n")
ini <- mice(df, maxit = 0, printFlag = FALSE)
meth <- ini$method
meth[c(normal_vars, non_normal_vars)] <- "pmm"
meth[binary_vars] <- "logreg"
meth[categorical_vars] <- "polyreg"

set.seed(123)
imputed_data <- mice(df, m = 5, maxit = 10, method = meth, printFlag = FALSE)

# Convert to long format for descriptive statistics
long_t1 <- complete(imputed_data, "long", include = FALSE)

# Initialize final table structure
table1_final <- data.frame(Variable = character(),
                           Death = character(),
                           Discharge = character(),
                           P_Value = character(),
                           stringsAsFactors = FALSE)

# ==============================================================================
# Normal variables (Mean ± SD with 2 decimals)
# ==============================================================================
table1_final <- rbind(table1_final, data.frame(Variable = "--- CONTINUOUS VARIABLES (NORMAL) ---", Death = "", Discharge = "", P_Value = ""))

for(var in normal_vars) {
  stats <- long_t1 %>%
    group_by(.imp, Status) %>%
    summarise(m = mean(.data[[var]], na.rm=TRUE), sd = sd(.data[[var]], na.rm=TRUE), .groups = 'drop') %>%
    group_by(Status) %>%
    summarise(final_m = mean(m), final_sd = mean(sd), .groups = 'drop')
  
  str_d   <- paste0(sprintf("%.2f", stats$final_m[stats$Status=="Death"]), " ± ", sprintf("%.2f", stats$final_sd[stats$Status=="Death"]))
  str_dis <- paste0(sprintf("%.2f", stats$final_m[stats$Status=="Discharge"]), " ± ", sprintf("%.2f", stats$final_sd[stats$Status=="Discharge"]))
  
  form_str <- paste0("as.numeric(", var, ") ~ Status")
  fit <- with(imputed_data, lm(as.formula(form_str)))
  p_val <- summary(pool(fit))$p.value[2]
  
  table1_final <- rbind(table1_final, data.frame(Variable = paste0("  ", var, " (Mean ± SD)"), 
                                                 Death = str_d, Discharge = str_dis, 
                                                 P_Value = sprintf("%.4f", p_val)))
}

# ==============================================================================
# Non-normal variables (Median [IQR] with 2 decimals)
# ==============================================================================
table1_final <- rbind(table1_final, data.frame(Variable = "--- CONTINUOUS VARIABLES (NON-NORMAL) ---", Death = "", Discharge = "", P_Value = ""))

for(var in non_normal_vars) {
  stats <- long_t1 %>%
    group_by(.imp, Status) %>%
    summarise(med = median(.data[[var]], na.rm=TRUE), 
              q1 = quantile(.data[[var]], 0.25, na.rm=TRUE), 
              q3 = quantile(.data[[var]], 0.75, na.rm=TRUE), .groups = 'drop') %>%
    group_by(Status) %>%
    summarise(final_med = mean(med), final_q1 = mean(q1), final_q3 = mean(q3), .groups = 'drop')
  
  str_d   <- paste0(sprintf("%.2f", stats$final_med[stats$Status=="Death"]), " [", sprintf("%.2f", stats$final_q1[stats$Status=="Death"]), " - ", sprintf("%.2f", stats$final_q3[stats$Status=="Death"]), "]")
  str_dis <- paste0(sprintf("%.2f", stats$final_med[stats$Status=="Discharge"]), " [", sprintf("%.2f", stats$final_q1[stats$Status=="Discharge"]), " - ", sprintf("%.2f", stats$final_q3[stats$Status=="Discharge"]), "]")
  
  form_str <- paste0("rank(as.numeric(", var, ")) ~ Status")
  fit <- with(imputed_data, lm(as.formula(form_str)))
  p_val <- summary(pool(fit))$p.value[2]
  
  table1_final <- rbind(table1_final, data.frame(Variable = paste0("  ", var, " (Median [IQR])"), 
                                                 Death = str_d, Discharge = str_dis, 
                                                 P_Value = sprintf("%.4f", p_val)))
}

# ==============================================================================
# Binary variables (n (%) with 2 decimal percentages)
# ==============================================================================
table1_final <- rbind(table1_final, data.frame(Variable = "--- BINARY VARIABLES ---", Death = "", Discharge = "", P_Value = ""))

for(var in binary_vars) {
  stats <- long_t1 %>%
    group_by(.imp, Status, .data[[var]]) %>%
    summarise(n = n(), .groups = 'drop_last') %>%
    mutate(pct = (n / sum(n)) * 100) %>%
    group_by(Status, .data[[var]]) %>%
    summarise(final_n = round(mean(n)), final_pct = mean(pct), .groups = 'drop')
  
  target_level <- levels(long_t1[[var]])[2] 
  
  n_d <- stats$final_n[stats$Status=="Death" & stats[[var]]==target_level]
  p_d <- stats$final_pct[stats$Status=="Death" & stats[[var]]==target_level]
  if(length(n_d)==0) { n_d <- 0; p_d <- 0 }
  
  n_dis <- stats$final_n[stats$Status=="Discharge" & stats[[var]]==target_level]
  p_dis <- stats$final_pct[stats$Status=="Discharge" & stats[[var]]==target_level]
  if(length(n_dis)==0) { n_dis <- 0; p_dis <- 0 }
  
  str_d   <- paste0(n_d, " (", sprintf("%.2f", p_d), "%)")
  str_dis <- paste0(n_dis, " (", sprintf("%.2f", p_dis), "%)")
  
  freq_tables <- lapply(1:5, function(m) { d <- complete(imputed_data, m); table(d[[var]], d$Status) })
  mean_table <- purrr::reduce(freq_tables, `+`) / 5
  
  if(any(mean_table < 5)) {
    cat(paste0("-> Low count for '", var, "'. Running Pooled Fisher's Exact Test (Median Method)...\n"))
    fisher_p_values <- sapply(1:5, function(m) {
      d <- complete(imputed_data, m)
      fisher.test(table(d[[var]], d$Status))$p.value
    })
    p_val <- median(fisher_p_values)
  } else {
    models_binary <- lapply(1:5, function(m) {
      d <- complete(imputed_data, m)
      glm(Status ~ factor(d[[var]]), data = d, family = binomial)
    })
    class(models_binary) <- "mira"
    p_val <- summary(pool(models_binary))$p.value[2]
  }
  
  table1_final <- rbind(table1_final, data.frame(Variable = paste0("  ", var, " (Level ", target_level, "), n (%)"), 
                                                 Death = str_d, Discharge = str_dis, 
                                                 P_Value = sprintf("%.4f", p_val)))
}

# ==============================================================================
# Multicategory Diagnosis variable (overall D1 test - 2 decimal percentages)
# ==============================================================================
table1_final <- rbind(table1_final, data.frame(Variable = "Diagnosis, n (%)", Death = "", Discharge = "", P_Value = ""))

diag_stats <- long_t1 %>%
  group_by(.imp, Status, Diagnosis) %>%
  summarise(n = n(), .groups = 'drop_last') %>%
  mutate(pct = (n / sum(n)) * 100) %>%
  group_by(Status, Diagnosis) %>%
  summarise(final_n = round(mean(n)), final_pct = mean(pct), .groups = 'drop')

for(lvl in levels(long_t1$Diagnosis)) {
  n_d   <- diag_stats$final_n[diag_stats$Status=="Death" & diag_stats$Diagnosis==lvl]
  p_d   <- diag_stats$final_pct[diag_stats$Status=="Death" & diag_stats$Diagnosis==lvl]
  n_dis <- diag_stats$final_n[diag_stats$Status=="Discharge" & diag_stats$Diagnosis==lvl]
  p_dis <- diag_stats$final_pct[diag_stats$Status=="Discharge" & diag_stats$Diagnosis==lvl]
  
  if(length(n_d)==0)   {n_d <- 0; p_d <- 0}
  if(length(n_dis)==0) {n_dis <- 0; p_dis <- 0}
  
  table1_final <- rbind(table1_final, data.frame(
    Variable = paste0("  - Level ", lvl),
    Death = paste0(n_d, " (", sprintf("%.2f", p_d), "%)"),
    Discharge = paste0(n_dis, " (", sprintf("%.2f", p_dis), "%)"),
    P_Value = ""
  ))
}

models_diag_alt <- lapply(1:5, function(m) { d <- complete(imputed_data, m); glm(Status ~ factor(Diagnosis), data = d, family = binomial) })
models_diag_null <- lapply(1:5, function(m) { d <- complete(imputed_data, m); glm(Status ~ 1, data = d, family = binomial) })
class(models_diag_alt) <- "mira"; class(models_diag_null) <- "mira"
p_val_diag <- D1(models_diag_alt, models_diag_null)$result[1, 4]

table1_final[table1_final$Variable == "Diagnosis, n (%)", "P_Value"] <- sprintf("%.4f", p_val_diag)

# ==============================================================================
# Add discretized clinical variables (percentages with 2 decimals)
# ==============================================================================
table1_final <- rbind(table1_final, data.frame(Variable = "--- CLINICAL CATEGORICAL VARIABLES (CUTOFFS) ---", Death = "", Discharge = "", P_Value = ""))

cutoffs <- list(
  Mean_Magnesium = list(breaks = c(-Inf, 1.8, 2.4, Inf), labels = c("1", "2", "3")),
  Mean_Phosphor  = list(breaks = c(-Inf, 2.5, 4.5, Inf), labels = c("1", "2", "3")),
  Mean_Calcium   = list(breaks = c(-Inf, 8.2, 9.3, Inf), labels = c("1", "2", "3")),
  Apache         = list(breaks = c(-Inf, 15, 24, Inf), labels = c("1", "2", "3")),
  BMI            = list(breaks = c(-Inf, 18.5, 24.9, 29.9, Inf), labels = c("1", "2", "3", "4")),
  Sofa_Mean      = list(breaks = c(-Inf, 4, 7, 10, Inf), labels = c("1", "2", "3" , "4")),
  Age            = list(breaks = c(-Inf, 44, 64, 79, Inf), labels = c("1", "2", "3", "4"))
)

for(var in names(cutoffs)) {
  cat_var <- paste0(var, "_cat")
  long_t1[[cat_var]] <- cut(long_t1[[var]], breaks = cutoffs[[var]]$breaks, labels = cutoffs[[var]]$labels)
  
  table1_final <- rbind(table1_final, data.frame(Variable = paste0(var, " (Categorical), n (%)"), Death = "", Discharge = "", P_Value = ""))
  
  cat_stats <- long_t1 %>%
    group_by(.imp, Status, .data[[cat_var]]) %>%
    summarise(n = n(), .groups = 'drop_last') %>%
    mutate(pct = (n / sum(n)) * 100) %>%
    group_by(Status, .data[[cat_var]]) %>%
    summarise(final_n = round(mean(n)), final_pct = mean(pct), .groups = 'drop')
  
  for(lvl in cutoffs[[var]]$labels) {
    n_d   <- cat_stats$final_n[cat_stats$Status=="Death" & cat_stats[[cat_var]]==lvl]
    p_d   <- cat_stats$final_pct[cat_stats$Status=="Death" & cat_stats[[cat_var]]==lvl]
    n_dis <- cat_stats$final_n[cat_stats$Status=="Discharge" & cat_stats[[cat_var]]==lvl]
    p_dis <- cat_stats$final_pct[cat_stats$Status=="Discharge" & cat_stats[[cat_var]]==lvl]
    
    if(length(n_d)==0)   {n_d <- 0; p_d <- 0}
    if(length(n_dis)==0) {n_dis <- 0; p_dis <- 0}
    
    table1_final <- rbind(table1_final, data.frame(
      Variable = paste0("  - Level ", lvl),
      Death = paste0(n_d, " (", sprintf("%.2f", p_d), "%)"),
      Discharge = paste0(n_dis, " (", sprintf("%.2f", p_dis), "%)"),
      P_Value = ""
    ))
  }
  
  models_alt <- lapply(1:5, function(m) {
    d <- complete(imputed_data, m)
    d[[cat_var]] <- cut(d[[var]], breaks = cutoffs[[var]]$breaks, labels = cutoffs[[var]]$labels)
    glm(Status ~ factor(d[[cat_var]]), data = d, family = binomial)
  })
  models_null <- lapply(1:5, function(m) {
    d <- complete(imputed_data, m)
    glm(Status ~ 1, data = d, family = binomial)
  })
  class(models_alt) <- "mira"; class(models_null) <- "mira"
  p_val_cat <- D1(models_alt, models_null)$result[1, 4]
  
  table1_final[table1_final$Variable == paste0(var, " (Categorical), n (%)"), "P_Value"] <- sprintf("%.4f", p_val_cat)
}

# ==============================================================================
# Final formatting of P-values, column headers, and Excel export
# ==============================================================================
table1_final$P_Value <- as.character(table1_final$P_Value)
table1_final$P_Value <- ifelse(table1_final$P_Value == "0.0000" | table1_final$P_Value == "0", "<0.001", table1_final$P_Value)

colnames(table1_final) <- c("Variable", 
                            paste0("Death (N = ", n_death, ")"), 
                            paste0("Discharge (N = ", n_discharge, ")"), 
                            "P_Value")

print(table1_final)
write.xlsx(table1_final, "Table1_Master_Report.xlsx", rowNames = FALSE)

-> Running MICE for Comprehensive Table 1...
-> Low count for 'Brain_Tumor'. Running Pooled Fisher's Exact Test (Median Method)...
                                           Variable       Death (N = 222)
1             --- CONTINUOUS VARIABLES (NORMAL) ---                      
2 

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 44: invalid start byte <traceback object at 0x00000242EC6DC640>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 16: invalid start byte <traceback object at 0x00000242EC6DC800>



3 

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 44: invalid start byte <traceback object at 0x00000242EC6DC600>
R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 16: invalid start byte <traceback object at 0x00000242EC6DCDC0>



4 

...

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 16: invalid start byte <traceback object at 0x00000242EC6FCCC0>


  0.0012
4 

R callback write-console: <class 'UnicodeDecodeError'> 'utf-8' codec can't decode byte 0xb1 in position 16: invalid start byte <traceback object at 0x00000242EC6FD300>


  0.1811
5                               
6     3.23 [2.65 - 3.92]  0.2695
7  56.50 [35.00 - 73.00]  <0.001
8     3.00 [1.98 - 5.00]  <0.001
9    9.40 [5.00 - 15.00]  <0.001
10                              
11          236 (34.20%)  0.0674
12          168 (24.35%)  0.4889
13          219 (31.74%)  0.0001
14          153 (22.17%)  0.0800
15            60 (8.70%)  0.0029
16            61 (8.84%)  0.0290
17            59 (8.55%)  0.0319
18            17 (2.46%)  0.1868
19          227 (32.90%)  <0.001
20                        <0.001
21          311 (45.07%)        
22          265 (38.41%)        
23          114 (16.52%)        
24                              
25                        0.0032
26          151 (21.83%)        
27          400 (58.00%)        
28          139 (20.17%)        
29                        0.0007
30          144 (20.84%)        
31          474 (68.75%)        
32           72 (10.41%)        
33                        <0.001
34          231 (33.54%)        
3


Attaching package: 'mice'

The following object is masked from 'package:stats':

    filter

The following objects are masked from 'package:base':

    cbind, rbind


Attaching package: 'dplyr'

The following objects are masked from 'package:stats':

    filter, lag

The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union

